In [192]:
pip install rapidfuzz

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [193]:
import pandas as pd

In [194]:
file_name='ingredients_BHUMI_2026-03-23.csv'
input_path=f'input_data/{file_name}'
output_path=f'output_data/{file_name}'

In [195]:
df = pd.read_csv(input_path)

In [196]:
df.head()

,Ingredient Name,Category,Unit of Measurement,Purchase Unit,Volume,Conversion Factor,Cost Per Unit,Cost Per Usage Unit,Description,Brand,...,Reorder Point,Storage Location,Storage Temperature,Is Perishable,Shelf Life Days,Active,Available,Barcode,SKU,Notes
0,MILK,DAIRY,LITRE,LITRE,1000 ML,1,48.00,48.00,NaN,SARAS,...,0.0,NaN,room_temperature,True,NaN,True,True,NaN,NaN,NaN
1,MILK TETRA,DAIRY,LITRE,LITRE,1000 ML,1,70.29,70.29,NaN,AMUL,...,0.0,NaN,room_temperature,False,NaN,True,True,NaN,NaN,NaN
2,CREAM,DAIRY,LITRE,LITRE,1000 ML,1,212.21,212.21,NaN,AMUL,...,0.0,NaN,room_temperature,False,NaN,True,True,NaN,NaN,NaN
3,BUTTER,DAIRY,KG,KG,500 GM,1,257.50,257.50,NaN,AMUL,...,0.0,NaN,room_temperature,False,NaN,True,True,NaN,NaN,NaN
4,MOZRELLA CHEESE,DAIRY,KG,KG,1000 GM,1,501.00,501.00,NaN,NaN,...,0.0,NaN,room_temperature,False,NaN,True,True,NaN,NaN,NaN


In [197]:
df["Ingredient Name"] = df["Ingredient Name"].str.title()

In [198]:
if "Volume" in df.columns and "Conversion Factor" in df.columns:
    # Remove old Conversion Factor first
    df = df.drop(columns=["Conversion Factor"])
    
    # Convert Volume → numeric
    df["Volume"] = df["Volume"].str.extract(r'(\d+)')[0].astype("Int64")
    
    # Rename Volume → Conversion Factor
    df = df.rename(columns={"Volume": "Conversion Factor"})

elif "Volume" in df.columns:
    # Convert then rename
    df["Volume"] = df["Volume"].str.extract(r'(\d+)')[0].astype("Int64")
    df = df.rename(columns={"Volume": "Conversion Factor"})

elif "Conversion Factor" in df.columns:
    # Already correct → do nothing
    pass

In [199]:
required_cols = [
    "Ingredient Name",
    "Unit of Measurement",
    "Purchase Unit",
    "Conversion Factor",
    "Cost Per Unit",
    "Category",
    "Description",
    "Minimum Stock Level",
    "Current Stock",
    "Storage Temperature",
    "Is Perishable",
    "Shelf Life Days",
    "Brand",
    "Active",
    "Available"
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

df = df[required_cols]

In [200]:
df.head()

,Ingredient Name,Unit of Measurement,Purchase Unit,Conversion Factor,Cost Per Unit,Category,Description,Minimum Stock Level,Current Stock,Storage Temperature,Is Perishable,Shelf Life Days,Brand,Active,Available
0,Milk,LITRE,LITRE,1000,48.00,DAIRY,NaN,108.0,0.0,room_temperature,True,NaN,SARAS,True,True
1,Milk Tetra,LITRE,LITRE,1000,70.29,DAIRY,NaN,NaN,0.0,room_temperature,False,NaN,AMUL,True,True
2,Cream,LITRE,LITRE,1000,212.21,DAIRY,NaN,1.0,0.0,room_temperature,False,NaN,AMUL,True,True
3,Butter,KG,KG,500,257.50,DAIRY,NaN,20.0,0.0,room_temperature,False,NaN,AMUL,True,True
4,Mozrella Cheese,KG,KG,1000,501.00,DAIRY,NaN,NaN,0.0,room_temperature,False,NaN,NaN,True,True


In [201]:
df.rename(columns={"Volume": "Conversion Factor"}, inplace=True)

In [202]:
df.head()

,Ingredient Name,Unit of Measurement,Purchase Unit,Conversion Factor,Cost Per Unit,Category,Description,Minimum Stock Level,Current Stock,Storage Temperature,Is Perishable,Shelf Life Days,Brand,Active,Available
0,Milk,LITRE,LITRE,1000,48.00,DAIRY,NaN,108.0,0.0,room_temperature,True,NaN,SARAS,True,True
1,Milk Tetra,LITRE,LITRE,1000,70.29,DAIRY,NaN,NaN,0.0,room_temperature,False,NaN,AMUL,True,True
2,Cream,LITRE,LITRE,1000,212.21,DAIRY,NaN,1.0,0.0,room_temperature,False,NaN,AMUL,True,True
3,Butter,KG,KG,500,257.50,DAIRY,NaN,20.0,0.0,room_temperature,False,NaN,AMUL,True,True
4,Mozrella Cheese,KG,KG,1000,501.00,DAIRY,NaN,NaN,0.0,room_temperature,False,NaN,NaN,True,True


In [203]:
category_keywords = {

    "Dairy": [
        "milk", "tetra", "cream", "butter", "cheese", "paneer",
        "lassi", "butter milk", "ice cream", "milk powder",
        "milk maid", "mascarpone"
    ],

    "Poultry": [
        "chicken", "egg"
    ],

    "Meat": [
        "mutton"
    ],

    "Seafood": [
        "fish", "prawn", "tuna"
    ],

    "Grains": [
        "atta", "wheat", "maida", "flour", "rice", "dalia",
        "suji", "rawa", "poha", "oats", "corn flakes",
        "pasta", "spaghetti", "vermicelli", "noodles"
    ],

    "Pulses": [
        "dal", "channa", "rajma", "lobia", "moong", "urad",
        "masur", "kabuli", "soya bean", "badi"
    ],

    "Fruits": [
        "mango", "apple", "banana", "pineapple", "orange",
        "guava", "litchi", "strawberry", "blueberry",
        "fruit", "kivi", "kokam"
    ],

    "Vegetables": [
        "garlic", "peas", "baby corn", "corn", "bamboo shoot",
        "mushroom", "jalepeno"
    ],

    "Spices": [
        "masala", "chilli", "haldi", "dhania", "jeera",
        "hing", "pepper", "dal chini", "elaichi", "tej pata",
        "kalonji", "long", "javetri", "rai", "amchur",
        "kesar", "saunf", "oregano", "chilli flakes",
        "seasoning","kasuri methi"
    ],

    "Condiments": [
        "sauce", "ketchup", "mayo", "mayonise", "vinegar",
        "pickle", "jam", "spread", "chutney", "paste"
    ],

    "Oils": [
        "oil", "ghee", "olive", "refined", "mustard"
    ],

    "Beverages": [
        "juice", "drink", "soda", "water", "cola",
        "coffee", "tea", "red bull", "lassi",
        "sharbat", "thandai", "tonic"
    ],

    "Syrup": [
        "syrup", "crush"
    ],

    "Dry goods": [
        "sugar", "salt", "baking", "powder", "custard",
        "yeast", "glucose", "pectin", "cocoa"
    ],

    "Nuts": [
        "almond", "badam", "kaju", "pista", "akhrot",
        "hazelnut"
    ],

    "Seeds": [
        "til"
    ],

    "Frozen": [
        "frozen", "french fry"
    ],

    "Canned": [
        "beans", "puree", "cocktail"
    ],

    "Bakery": [
        "bread", "biscuit", "cake", "wafer",
        "chocolate", "custard"
    ],

    "Herbs": [
        "basil", "mint", "coriander leaf", "parsley", "rosemary",
        "thyme", "dill", "fennel", "bay leaf"
    ],

    "Kitchen": [
        "ajinomoto", "eno", "food color", "gel", "dust"
    ],

    "Disposal": [
        "paper", "foil", "container", "tray", "cup",
        "plate", "spoon", "bag", "box", "tissue",
        "straw", "stirrer", "packing"
    ],

    "House keeping": [
        "phenyl", "bleaching", "soap", "surf", "taski",
        "cleaner", "detergent", "mop", "brush", "scrub",
        "handwash", "gloves"
    ],

    "Room supply": [
        "shampoo", "lotion", "slipper", "dental kit",
        "shaving", "comb", "body wash", "soap"
    ],

    "Stationery": [
        "pen", "paper", "register", "file", "marker",
        "stapler", "book", "voucher", "tag"
    ],

    "Other": [
        "gas", "coal", "battery", "agarbatti", "machine"
    ]
}

In [204]:
from rapidfuzz import process, fuzz

def map_category_fuzzy(name, category_keywords, threshold=70):
    name = str(name).lower()

    best_match = None
    best_score = 0
    best_category = "Other"

    for category, keywords in category_keywords.items():
        match, score, _ = process.extractOne(
            name,
            keywords,
            scorer=fuzz.partial_ratio
        )

        if score > best_score:
            best_score = score
            best_match = match
            best_category = category

    if best_score >= threshold:
        return best_category
    else:
        return "Other"

In [205]:
df["Category"] = df["Ingredient Name"].apply(
    lambda x: map_category_fuzzy(x, category_keywords)
)

In [206]:
unit_keywords = {
    "bottle": ["ml", "ltr", "liter", "litre", "water", "drink", "juice", "soda","bottle",'BOTTLE'],
    
    "packet": ["gm", "kg", "gram", "gms", "powder", "flour", "atta", "rice", "dal",'packet'],
    
    "box": ["box", "cake box", "pizza box", "pastry box"],
    
    "carton": ["tetra", "carton"],
    
    "container": ["container", "dabba", "tray", "cup"],
    
    "pouch": ["pouch", "sachet"],
    
    "can": ["can", "tin"],
    
    "jar": ["jar"],
    
    "bag": ["bag", "carry bag"],
    
    "roll": ["roll"],
    
    "sheet": ["sheet"],
    
    "bundle": ["bundle"],
    
    "individual": ["piece", "pc", "nos", "unit"],
    
    "crate": ["crate"],
    
    "drum": ["drum"],
    
    "barrel": ["barrel"],
    
    "tank": ["tank"],
    
    "pair": ["pair"],
    
    "case": ["case"],
    
    "pallet": ["pallet"],
    
    "sack": ["sack"]
}

In [207]:
from rapidfuzz import process, fuzz

def map_unit_smart(value, unit_keywords, threshold=75):
    value = str(value).lower()

    # 1. Direct keyword match (FAST + ACCURATE)
    for unit, keywords in unit_keywords.items():
        if any(k in value for k in keywords):
            return unit

    # 2. Fuzzy fallback
    best_score = 0
    best_unit = "individual"

    words = value.split()

    for unit, keywords in unit_keywords.items():
        for word in words:
            match, score, _ = process.extractOne(
                word,
                keywords,
                scorer=fuzz.partial_ratio
            )

            if score > best_score:
                best_score = score
                best_unit = unit

    return best_unit if best_score >= threshold else "individual"

In [208]:
df["Purchase Unit"] = df["Purchase Unit"].apply(
    lambda x: map_unit_smart(x, unit_keywords)
)

In [209]:
df.head()

,Ingredient Name,Unit of Measurement,Purchase Unit,Conversion Factor,Cost Per Unit,Category,Description,Minimum Stock Level,Current Stock,Storage Temperature,Is Perishable,Shelf Life Days,Brand,Active,Available
0,Milk,LITRE,bottle,1000,48.00,Dairy,NaN,108.0,0.0,room_temperature,True,NaN,SARAS,True,True
1,Milk Tetra,LITRE,bottle,1000,70.29,Dairy,NaN,NaN,0.0,room_temperature,False,NaN,AMUL,True,True
2,Cream,LITRE,bottle,1000,212.21,Dairy,NaN,1.0,0.0,room_temperature,False,NaN,AMUL,True,True
3,Butter,KG,packet,500,257.50,Dairy,NaN,20.0,0.0,room_temperature,False,NaN,AMUL,True,True
4,Mozrella Cheese,KG,packet,1000,501.00,Dairy,NaN,NaN,0.0,room_temperature,False,NaN,NaN,True,True


In [210]:
unit_synonyms = {
    "kg": ["kg", "kgs", "kilogram", "kilo", "k.g"],
    "gram": ["g", "gm", "gms", "gram", "grams"],
    
    "liter": ["l", "ltr", "ltrs", "liter", "litre", "liters", "litres"],
    "ml": ["ml", "milliliter", "millilitre"],
    
    "packet": ["packet", "pkt", "pack", "pck"],
    "pouch": ["pouch", "sachet"],
    
    "bottle": ["bottle", "btl", "bottel"],
    "box": ["box", "bx"],
    "can": ["can", "tin"],
    "jar": ["jar"],
    
    "container": ["container", "cont", "dabba"],
    
    "piece": ["piece", "pc", "pcs", "nos", "unit"],
    "pair": ["pair"],
    "dozen": ["dozen"],
    
    "cup": ["cup"],
    "tablespoon": ["tbsp", "tablespoon"],
    "teaspoon": ["tsp", "teaspoon"],
    
    "ounce": ["oz", "ounce"],
    "pound": ["lb", "pound"],
    
    "gallon": ["gallon"],
    "quart": ["quart"],
    "pint": ["pint"],
    "fluid_ounce": ["fl oz"],
    
    "roll": ["roll"],
    "bunch": ["bunch"]
}

In [211]:
from rapidfuzz import process, fuzz
import re

def normalize_unit(value, threshold=80):
    value = str(value).lower().strip()

    # 🔹 remove numbers (100 ml → ml)
    value = re.sub(r'\d+', '', value).strip()

    # 🔹 remove special chars
    value = re.sub(r'[^a-zA-Z\s]', '', value).strip()

    # -------------------------
    # 1. DIRECT MATCH (BEST)
    # -------------------------
    for standard_unit, synonyms in unit_synonyms.items():
        if any(word in value for word in synonyms):
            return standard_unit

    # -------------------------
    # 2. FUZZY MATCH (BACKUP)
    # -------------------------
    all_words = [w for words in unit_synonyms.values() for w in words]

    match, score, _ = process.extractOne(
        value,
        all_words,
        scorer=fuzz.ratio
    )

    if score >= threshold:
        for standard_unit, synonyms in unit_synonyms.items():
            if match in synonyms:
                return standard_unit

    # -------------------------
    # 3. FALLBACK
    # -------------------------
    return "piece"

In [212]:
df["Unit of Measurement"] = df["Unit of Measurement"].apply(normalize_unit)

In [213]:
df.to_csv(output_path, index=False)